# Исследование продаж и оценок видеоигры 2000-2013 гг.

- Автор: Суздалов Дмитрий
- Дата: 01.11.2026

### Цели и задачи проекта
Задача - познакомиться с данными, проверить их корректность и провести предобработку, получив необходимый срез данных.
Перед анализом данных необходимо: 
1. Отобрать данные по времени выхода игры. Вам нужен период с 2000 по 2013 год включительно.
2. Категоризовать игры по оценкам пользователей и экспертов. Выделите три категории:
    - высокая оценка — с оценкой от 8 до 10 и от 80 до 100, включая правые границы интервалов.
    - средняя оценка — с оценкой от 3 до 8 и от 30 до 80, не включая правые границы интервалов.
    - низкая оценка — с оценкой от 0 до 3 и от 0 до 30, не включая правые границы интервалов.
3. Выделить топ-7 платформ по количеству игр, выпущенных за весь требуемый период.

### Описание данных

Данные new_games.csv содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:
- `Name` — название игры.
- `Platform` — название платформы.
- `Year of Release` — год выпуска игры.
- `Genre` — жанр игры.
- `NA sales` — продажи в Северной Америке (в миллионах проданных копий).
- `EU sales` — продажи в Европе (в миллионах проданных копий).
- `JP sales` — продажи в Японии (в миллионах проданных копий).
- `Other sales` — продажи в других странах (в миллионах проданных копий).
- `Critic Score` — оценка критиков (от 0 до 100).
- `User Score` — оценка пользователей (от 0 до 10).
- `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). 
Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта


* Загрузка данных и знакомство
* Предобработка данных
* Фильтрация
* Категоризация
* Вывод
---

## 1. Загрузка данных и знакомство с ними

- Загрузите необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


In [1]:
import pandas as pd

df = pd.read_csv('C:/Users/Дмитрий/Desktop/Python/new_games.csv')

In [2]:
#Выведем информацию
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB
None


Из вывода метода `info()` видно, что почти в каждом столбце присутствуют пропуски

In [3]:
#Воспользуемся функцией `display()` чтобы посмотреть на датафрейм нагляднее:
display(df)

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16951,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,0.00,0.0,0.01,0.00,NaN,NaN,NaN
16952,LMA Manager 2007,X360,2006.0,Sports,0.00,0.01,0.0,0.00,NaN,NaN,NaN
16953,Haitaka no Psychedelica,PSV,2016.0,Adventure,0.00,0.0,0.01,0.00,NaN,NaN,NaN
16954,Spirits & Spells,GBA,2003.0,Platform,0.01,0.0,0.0,0.00,NaN,NaN,NaN


In [4]:
#Выведем первые строки датафрейма:
print(df.head())

                       Name Platform  Year of Release         Genre  NA sales  \
0                Wii Sports      Wii           2006.0        Sports     41.36   
1         Super Mario Bros.      NES           1985.0      Platform     29.08   
2            Mario Kart Wii      Wii           2008.0        Racing     15.68   
3         Wii Sports Resort      Wii           2009.0        Sports     15.61   
4  Pokemon Red/Pokemon Blue       GB           1996.0  Role-Playing     11.27   

  EU sales JP sales  Other sales  Critic Score User Score Rating  
0    28.96     3.77         8.45          76.0          8      E  
1     3.58     6.81         0.77           NaN        NaN    NaN  
2    12.76     3.79         3.29          82.0        8.3      E  
3    10.93     3.28         2.95          80.0          8      E  
4     8.89    10.22         1.00           NaN        NaN    NaN  


Из данных следует, что всего у нас 16956 строк, во многих присутствуют пропуски. Больше всего пропусков в строке Rating и Score's. Так же, на удивление у столбца User Score тип данных Str, что не очень удобно, учитывая что там числа, но поправимо. Есть пропуски и в столбце Name, что никак не исправить, придется эти строки, вероятно, убрать, благо их всего 2. Также, возвращаясь к типам данных столбцы EU/JP Sales тоже в формате Str, а Year of Release = float64, будем исправлять

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма


Выведу на экран названия всех столбцов датафрейма и проверю их стиль написания.
Приведу все столбцы к стилю snake_case. Названия должны быть в нижнем регистре, а вместо пробелов — подчёркивания.

In [5]:
#Выводим на экран названия столбцов
print(df.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')


In [6]:
#Делаем преобразвания:
new_columns = df.columns.str.lower()
new_columns = new_columns.str.replace(' ', '_')
df.columns = new_columns
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')


Получилось привести столбцы к общему виду, двигаемся дальше

### 2.2. Типы данных



Проведу преобразование типов данных.

In [7]:
#выведем вновь датафрейм:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16956 non-null  str    
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  str    
 4   na_sales         16956 non-null  float64
 5   eu_sales         16956 non-null  str    
 6   jp_sales         16956 non-null  str    
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       10152 non-null  str    
 10  rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB
None


Будем делать преобразование постепенно

In [8]:
print(df.nunique())

name               11559
platform              31
year_of_release       37
genre                 24
na_sales             402
eu_sales             308
jp_sales             245
other_sales          155
critic_score          82
user_score            96
rating                 8
dtype: int64


In [9]:
print(df.isna().sum())

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8714
user_score         6804
rating             6871
dtype: int64


In [10]:
#считаем кол-во пропусков в относительных значениях
print(df.isna().mean())

name               0.000118
platform           0.000000
year_of_release    0.016218
genre              0.000118
na_sales           0.000000
eu_sales           0.000000
jp_sales           0.000000
other_sales        0.000000
critic_score       0.513918
user_score         0.401274
rating             0.405225
dtype: float64


Из первичного анализа видно, что у нас в датафрейме присутствуют два пропуска в name и genre. Так как их всего лишь суммарно 4 на датафрейм из 16+ тысяч строк считаю от них можно избавиться:

In [11]:
df = df.dropna(subset = ['name', 'genre'])
print(df.isna().sum())

name                  0
platform              0
year_of_release     275
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8712
user_score         6802
rating             6869
dtype: int64


Теперь по очереди проверим sales:

In [12]:
print(df['na_sales'].unique())
print(df['eu_sales'].unique())
print(df['jp_sales'].unique())
print(df['other_sales'].unique())

[4.136e+01 2.908e+01 1.568e+01 1.561e+01 1.127e+01 2.320e+01 1.128e+01
 1.396e+01 1.444e+01 2.693e+01 9.050e+00 9.710e+00 9.000e+00 8.920e+00
 1.500e+01 9.010e+00 7.020e+00 9.430e+00 1.278e+01 4.740e+00 6.380e+00
 1.083e+01 9.540e+00 9.660e+00 8.410e+00 6.060e+00 3.430e+00 5.510e+00
 6.850e+00 9.040e+00 5.890e+00 6.030e+00 9.700e+00 5.280e+00 4.990e+00
 8.250e+00 8.520e+00 5.540e+00 6.990e+00 6.620e+00 5.030e+00 5.990e+00
 3.960e+00 2.500e+00 7.970e+00 6.910e+00 4.340e+00 4.350e+00 3.010e+00
 6.160e+00 6.760e+00 4.020e+00 4.890e+00 2.960e+00 4.760e+00 5.010e+00
 6.730e+00 5.950e+00 3.660e+00 5.550e+00 7.040e+00 6.650e+00 3.880e+00
 5.800e+00 4.100e+00 5.930e+00 4.360e+00 5.700e+00 2.030e+00 4.400e+00
 5.050e+00 3.540e+00 1.120e+00 6.820e+00 1.750e+00 3.740e+00 1.060e+00
 2.790e+00 2.910e+00 9.900e-01 2.570e+00 2.990e+00 2.280e+00 7.280e+00
 2.760e+00 2.900e+00 2.810e+00 6.600e-01 3.780e+00 3.270e+00 3.560e+00
 5.390e+00 4.590e+00 4.810e+00 4.460e+00 3.480e+00 2.850e+00 2.530e+00
 2.980

В eu_sales и jp_sales значения тоже числа либо unknown и записаны как str, посчитаем сколько их по очереди:

In [13]:
#тест
unknown_eu_sales = df[df['eu_sales'] == 'unknown'].count()
print(unknown_eu_sales)

name               6
platform           6
year_of_release    6
genre              6
na_sales           6
eu_sales           6
jp_sales           6
other_sales        6
critic_score       5
user_score         5
rating             5
dtype: int64


In [14]:
#тест
unknown_jp_sales = df[df['jp_sales'] == 'unknown'].count()
print(unknown_jp_sales)

name               4
platform           4
year_of_release    4
genre              4
na_sales           4
eu_sales           4
jp_sales           4
other_sales        4
critic_score       3
user_score         4
rating             4
dtype: int64


Получилось, что у unknown_eu_sales = 6, а unknown_jp_sales = 4, избавимся и от них:

In [15]:
df = df[df['eu_sales'] != 'unknown']
df = df[df['jp_sales'] != 'unknown'] 
df['eu_sales'] = pd.to_numeric(df['eu_sales'])
df['jp_sales'] = pd.to_numeric(df['jp_sales'])
print(df.info())

<class 'pandas.DataFrame'>
Index: 16944 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16944 non-null  str    
 1   platform         16944 non-null  str    
 2   year_of_release  16669 non-null  float64
 3   genre            16944 non-null  str    
 4   na_sales         16944 non-null  float64
 5   eu_sales         16944 non-null  float64
 6   jp_sales         16944 non-null  float64
 7   other_sales      16944 non-null  float64
 8   critic_score     8234 non-null   float64
 9   user_score       10143 non-null  str    
 10  rating           10076 non-null  str    
dtypes: float64(6), str(5)
memory usage: 1.6 MB
None


Теперь займемся нулевыми значениями в Score's:

In [16]:
#тест
print(df['user_score'].unique())

<StringArray>
[  '8',   nan, '8.3', '8.5', '6.6', '8.4', '8.6', '7.7', '6.3', '7.4', '8.2',
   '9', '7.9', '8.1', '8.7', '7.1', '3.4', '5.3', '4.8', '3.2', '8.9', '6.4',
 '7.8', '7.5', '2.6', '7.2', '9.2',   '7', '7.3', '4.3', '7.6', '5.7',   '5',
 '9.1', '6.5', 'tbd', '8.8', '6.9', '9.4', '6.8', '6.1', '6.7', '5.4',   '4',
 '4.9', '4.5', '9.3', '6.2', '4.2',   '6', '3.7', '4.1', '5.8', '5.6', '5.5',
 '4.4', '4.6', '5.9', '3.9', '3.1', '2.9', '5.2', '3.3', '4.7', '5.1', '3.5',
 '2.5', '1.9',   '3', '2.7', '2.2',   '2', '9.5', '2.1', '3.6', '2.8', '1.8',
 '3.8',   '0', '1.6', '9.6', '2.4', '1.7', '1.1', '0.3', '1.5', '0.7', '1.2',
 '2.3', '0.5', '1.3', '0.2', '0.6', '1.4', '0.9',   '1', '9.7']
Length: 97, dtype: str


Есть неудобные значения:

In [17]:
df['user_score'] = pd.to_numeric(df['user_score'], errors = 'coerce')
print(df.info())

<class 'pandas.DataFrame'>
Index: 16944 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16944 non-null  str    
 1   platform         16944 non-null  str    
 2   year_of_release  16669 non-null  float64
 3   genre            16944 non-null  str    
 4   na_sales         16944 non-null  float64
 5   eu_sales         16944 non-null  float64
 6   jp_sales         16944 non-null  float64
 7   other_sales      16944 non-null  float64
 8   critic_score     8234 non-null   float64
 9   user_score       7680 non-null   float64
 10  rating           10076 non-null  str    
dtypes: float64(7), str(4)
memory usage: 1.6 MB
None


In [18]:
print(df.isna().sum())


name                  0
platform              0
year_of_release     275
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8710
user_score         9264
rating             6868
dtype: int64


### 2.3. Наличие пропусков в данных




Посчитаю количество пропусков в каждом столбце в абсолютных и относительных значениях.

In [19]:
#считаем кол-во пропусков в абсолютных значениях
print(df.isna().sum())

name                  0
platform              0
year_of_release     275
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8710
user_score         9264
rating             6868
dtype: int64


In [20]:
#считаем кол-во пропусков в относительных значениях
print(df.isna().mean())

name               0.000000
platform           0.000000
year_of_release    0.016230
genre              0.000000
na_sales           0.000000
eu_sales           0.000000
jp_sales           0.000000
other_sales        0.000000
critic_score       0.514046
user_score         0.546742
rating             0.405335
dtype: float64


Больше всего характерны пропуски были для столбцов Score's и rating. Вероятно, в выборку попали малоизвестные игры, которые никто не обозревал и рейтинг им никто из издателей не выставлял. Либо, основываясь на данных, предоставленным в описании к работе, игры вышли до 1994 года, потому рейтинга у них нет, проверим:

In [21]:
list_of_years = df[df['rating'].isna()]['year_of_release'].tolist()
count_games = 0

for x in list_of_years:
    if x < 1994:
        count_games += 1
print(count_games)


366


Гипотеза подтвердилась, но этого не достаточно, чтобы все объяснить. 

Также пропуски есть в year_of_release, вероятно тоже малоизвестные игры с непонятным происхождением.



В итоге у нас остались пропуски лишь в year_of_release, critic_score, user_score и rating. Как будто можно было бы заполнить данные rating по жанру и платформе, но игры могут слишком сильно различаться, потому это будет слишком необъективно и вообще навредит данным, оставим как есть.

In [22]:
print(df.isna().sum())

name                  0
platform              0
year_of_release     275
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8710
user_score         9264
rating             6868
dtype: int64


### 2.4. Явные и неявные дубликаты в данных


In [23]:
#Выведу информацию для удобства
print(df.info())

<class 'pandas.DataFrame'>
Index: 16944 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16944 non-null  str    
 1   platform         16944 non-null  str    
 2   year_of_release  16669 non-null  float64
 3   genre            16944 non-null  str    
 4   na_sales         16944 non-null  float64
 5   eu_sales         16944 non-null  float64
 6   jp_sales         16944 non-null  float64
 7   other_sales      16944 non-null  float64
 8   critic_score     8234 non-null   float64
 9   user_score       7680 non-null   float64
 10  rating           10076 non-null  str    
dtypes: float64(7), str(4)
memory usage: 1.6 MB
None


Начнем изучение дубликатов по очереди:

In [24]:
print(df.duplicated().sum())

182


Датафрейм содержит 182 полных дубликата, избавимся от них:

In [25]:
df.drop_duplicates(subset = None, keep = 'first', inplace = True)
print(df.info())

<class 'pandas.DataFrame'>
Index: 16762 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16762 non-null  str    
 1   platform         16762 non-null  str    
 2   year_of_release  16490 non-null  float64
 3   genre            16762 non-null  str    
 4   na_sales         16762 non-null  float64
 5   eu_sales         16762 non-null  float64
 6   jp_sales         16762 non-null  float64
 7   other_sales      16762 non-null  float64
 8   critic_score     8161 non-null   float64
 9   user_score       7611 non-null   float64
 10  rating           9977 non-null   str    
dtypes: float64(7), str(4)
memory usage: 1.5 MB
None


C:\Users\Дмитрий\AppData\Local\Temp\ipykernel_8068\1936444835.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(subset = None, keep = 'first', inplace = True)


In [26]:
print(df.duplicated(subset = ['name', 'platform']).sum())

63


В датафрейме присутствуют и дубликаты вот такой комбинации name & platform, избавляемся:

In [27]:
df.drop_duplicates(subset = ['name', 'platform'], keep = 'first', inplace = True)
print(df.info())

<class 'pandas.DataFrame'>
Index: 16699 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16699 non-null  str    
 1   platform         16699 non-null  str    
 2   year_of_release  16431 non-null  float64
 3   genre            16699 non-null  str    
 4   na_sales         16699 non-null  float64
 5   eu_sales         16699 non-null  float64
 6   jp_sales         16699 non-null  float64
 7   other_sales      16699 non-null  float64
 8   critic_score     8125 non-null   float64
 9   user_score       7578 non-null   float64
 10  rating           9936 non-null   str    
dtypes: float64(7), str(4)
memory usage: 1.5 MB
None


C:\Users\Дмитрий\AppData\Local\Temp\ipykernel_8068\180673195.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(subset = ['name', 'platform'], keep = 'first', inplace = True)


Теперь проверим неявные дубликаты:

In [28]:
print(df['genre'].unique())

<StringArray>
[      'Sports',     'Platform',       'Racing', 'Role-Playing',
       'Puzzle',         'Misc',      'Shooter',   'Simulation',
       'Action',     'Fighting',    'Adventure',     'Strategy',
         'MISC', 'ROLE-PLAYING',       'RACING',       'ACTION',
      'SHOOTER',     'PLATFORM',     'FIGHTING',       'SPORTS',
   'SIMULATION',       'PUZZLE',    'ADVENTURE']
Length: 23, dtype: str


Уже видно, что есть неявные дубликаты: 'Platform' и 'PLATFORM' означают одно и то же. В таком случае для столбцов name, platform, genre и даже rating приведу к общему виду. К нижнему регистру, а для rating наоборот верхний регистр:

In [29]:
df['name'] = df['name'].str.lower().str.strip()
df['platform'] = df['platform'].str.lower().str.strip()
df['genre'] = df['genre'].str.lower().str.strip()
df['rating'] = df['rating'].str.upper().str.strip()
print(df.duplicated(subset = ['name', 'platform', 'genre', 'rating']).sum())

0


Окак, после преобразований дубликатов не оказалось! В таком случае двигаемся дальше

In [30]:
#Изначально было 16956 строк данных, буду использовать это число
#Посчитаем абсолютное количество удаленных строк:
first_results_absolut = 16956 - df.shape[0]
print(f"Абсолютное количество удаленных строк: {first_results_absolut}")
first_results_share = round((first_results_absolut / 16956 * 100), 2)
print(f"Доля удаленных строк: {first_results_share}")


Абсолютное количество удаленных строк: 257
Доля удаленных строк: 1.52


Получилось удалить всего 1,52% от общего количества строк, что совсем немного

---

## 3. Фильтрация данных

Задание: "Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберите данные по этому показателю. Сохраните новый срез данных в отдельном датафрейме, например `df_actual`".

Приступаем:

In [31]:
df_actual = df.copy()
df_actual = df_actual[(df_actual['year_of_release'] >= 2000) & (df_actual['year_of_release'] <= 2013)]
print(df_actual.info())
#Проверим
print(df_actual['year_of_release'].min())
print(df_actual['year_of_release'].max())

<class 'pandas.DataFrame'>
Index: 12769 entries, 0 to 16954
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12769 non-null  str    
 1   platform         12769 non-null  str    
 2   year_of_release  12769 non-null  float64
 3   genre            12769 non-null  str    
 4   na_sales         12769 non-null  float64
 5   eu_sales         12769 non-null  float64
 6   jp_sales         12769 non-null  float64
 7   other_sales      12769 non-null  float64
 8   critic_score     7159 non-null   float64
 9   user_score       6473 non-null   float64
 10  rating           8712 non-null   str    
dtypes: float64(7), str(4)
memory usage: 1.2 MB
None
2000.0
2013.0


В df_actual попало 12769 строк, подходящих под условие

---

## 4. Категоризация данных
    
Проведу категоризацию данных:
- Разделю все игры по оценкам пользователей и выделю такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [32]:
#Для данной задачи создадим функцию:
def game_score_category(score):
    if pd.isna(score):
        return "Нет оценки"
    
    if score < 3:
        return "Низкая оценка"
    elif score < 8:
        return "Средняя оценка"
    else:
        return "Высокая оценка"

df_actual['user_game_score_category'] = df_actual['user_score'].apply(game_score_category)

print(df_actual['user_game_score_category'].unique())

<StringArray>
['Высокая оценка', 'Средняя оценка', 'Нет оценки', 'Низкая оценка']
Length: 4, dtype: str


- Разделю все игры по оценкам критиков и выделите такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [33]:
#Для данной задачи создадим функцию:
def game_score_category_critics(score):
    if pd.isna(score):
        return "Нет оценки"
    
    if score < 30:
        return "Низкая оценка"
    elif score < 80:
        return "Средняя оценка"
    else:
        return "Высокая оценка"

df_actual['critics_game_score_category'] = df_actual['critic_score'].apply(game_score_category_critics)

print(df_actual['critics_game_score_category'].unique())

<StringArray>
['Средняя оценка', 'Высокая оценка', 'Нет оценки', 'Низкая оценка']
Length: 4, dtype: str


In [34]:
#Проверим
print(df_actual['critics_game_score_category'].unique())
print(df_actual['user_game_score_category'].unique())

<StringArray>
['Средняя оценка', 'Высокая оценка', 'Нет оценки', 'Низкая оценка']
Length: 4, dtype: str
<StringArray>
['Высокая оценка', 'Средняя оценка', 'Нет оценки', 'Низкая оценка']
Length: 4, dtype: str


Теперь проверю результат: сгруппирую данные по выделенным категориям и посчитаю количество игр в каждой категории.

In [35]:
print(df_actual['user_game_score_category'].value_counts())

user_game_score_category
Нет оценки        6296
Средняя оценка    4077
Высокая оценка    2280
Низкая оценка      116
Name: count, dtype: int64


In [36]:
print(df_actual['critics_game_score_category'].value_counts())

critics_game_score_category
Нет оценки        5610
Средняя оценка    5421
Высокая оценка    1683
Низкая оценка       55
Name: count, dtype: int64


Средних оценок оказалось больше всего

Теперь выделю топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [37]:
top_platforms = df_actual['platform'].value_counts()[:7]
print(top_platforms)


platform
ps2     2126
ds      2117
wii     1275
psp     1179
x360    1117
ps3     1086
gba      810
Name: count, dtype: int64


Самая популярная платформа: ps2

---

## 5. Итоговый вывод

В конце напишите основной вывод и отразите, какую работу проделали. Не забудьте указать описание среза данных и новых полей, которые добавили в исходный датасет.

В ходе работы были проведены множественные операции с датафреймом:
* Загрузка данных
* Предобработка(столбцы были приведены к общему стилю, исправлены типы данных. Также были исправлены пропуски и убраны дубликаты)
* Всего было удалено 257 строк (1,52%) данных

Также уже обработанные данные были превращены в новый датафрейм `df_actual` со срезом данных с 2000 по 2013 год. Всего срез составил 12769 строк. 
В новый датафрейм были добавлены новые столбцы-категории: `user_game_score_category` и `critics_game_score_category`, созданные на основе столбцов `user_score` и `critis_score` соответственно. 

В ходе исследования нового датафрейма выяснилось, что в данный период среза самыми популярными платформами стали:
* ps2
* ds
* wii
* psp
* x360
* ps3
* gba
